In [1]:
import torch
from diffusers import DiffusionPipeline
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available else "cpu"

In [2]:
model_id = "black-forest-labs/FLUX.1-dev"
pipeline = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)

model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

...

(…)ion_pytorch_model.safetensors.index.json:   0%|          | 0.00/121k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/820 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
from diffusers import FluxPipeline, AutoPipelineForText2Image,
from diffusers import FluxTransformer2DModel, BitsAndBytesConfig
from transformers import T5EncoderModel
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig
import torch
import gc
import matplotlib.pyplot as plt

ckpt_id = "black-forest-labs/FLUX.1-dev"
fused_transformer_path = "fused_transformer"

# тип данных, который будем использовать
bnb_4bit_compute_dtype = torch.float16

# конфиг для загрузки облегчённой версии основной модели
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=bnb_4bit_compute_dtype,
)

# загружаем облегчённую версию основной модели
transformer = FluxTransformer2DModel.from_pretrained(
    ckpt_id, subfolder="transformer",
    quantization_config=nf4_config, torch_dtype=torch.float16
)

# конфиг для загрузки облегчённой версии тектового энкодера
quant_config = TransformersBitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.float16
)

# загружаем текстовый энкодер
text_encoder = T5EncoderModel.from_pretrained(
    ckpt_id, 
    subfolder="text_encoder_2", 
    quantization_config=quant_config, 
    torch_dtype=torch.float16,
)

# создаём полный пайплайн
pipeline = FluxPipeline.from_pretrained(
    ckpt_id,
    transformer=transformer,
    text_encoder_2=text_encoder,
    torch_dtype=bnb_4bit_compute_dtype,
)

# удаляем лишнее
del text_encoder
del transformer

# и чистим кеш 
gc.collect()
torch.cuda.empty_cache()

pipeline.to("cuda")

In [ ]:
prompt = (
    "pixel art of futuristic stormtrooper with glossy white armor and a sleek helmet,"
    " standing heroically on a lush alien planet, vibrant flowers blooming around, soft"
    " sunlight illuminating the scene, a gentle breeze rustling the leaves"
)

image = pipeline(
    prompt=prompt,
    num_inference_steps=30,
    width=512,
    height=384,
    guidance_scale=3.5,
).images[0]

plt.imshow(image)